# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing dataset entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
<br>
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata  # metadata is an object (not dict), so use attributes
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Number of record sets:", len(metadata.record_sets))

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

In [ ]:
# List all record sets in this dataset, display their @id, name, and description
for rs in metadata.record_sets:
    print(f"Record Set @id: {rs.id}, Name: {rs.name}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, Name: {field.name}, DataType: {field.data_type if hasattr(field, 'data_type') else 'N/A'}")
        # If the field is column-backed, list columns
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"      [Column] @id: {col.id}, Name: {col.name}")
    print("---")
# For further operations, get the @id of the main record set
main_record_set_id = metadata.record_sets[0].id if metadata.record_sets else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
All record and field references are by `@id`.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))  # yields dicts
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for RecordSet @id '{rs_id}' with shape {df.shape}")

# Display columns (by @id) from main record set
if main_record_set_id:
    print("Columns of main record set:", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
All entity references are via `@id`.

In [ ]:
# Identify numeric fields from main record set
rs = [r for r in metadata.record_sets if r.id == main_record_set_id][0]
numeric_fields = [f for f in rs.fields if hasattr(f, 'data_type') and f.data_type in ['Integer', 'Float', 'Number']]

# Use the first numeric field for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0].id
else:
    numeric_field_id = dataframes[main_record_set_id].select_dtypes('number').columns[0] if not dataframes[main_record_set_id].select_dtypes('number').empty else None

threshold = 10
df_main = dataframes[main_record_set_id]
if numeric_field_id and numeric_field_id in df_main.columns:
    filtered_df = df_main[df_main[numeric_field_id] > threshold]
    print(f"Filtered records with field @id '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())
    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical field
    group_fields = [f for f in rs.fields if hasattr(f, 'data_type') and f.data_type == 'Text']
    group_field_id = group_fields[0].id if group_fields else None
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped (mean) data by field @id '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
# Visualize numeric field distribution
if numeric_field_id and numeric_field_id in df_main.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df_main[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in main record set")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # If group field exists, plot boxplot
    if group_field_id and group_field_id in df_main.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df_main[group_field_id], y=df_main[numeric_field_id])
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded the FAIR^2 dataset using `mlcroissant` by referencing Croissant schema and entities by `@id`.
- Identified the main record set and explored both numeric and categorical fields.
- Applied filtering, normalization, and grouping operations for exploratory analysis.
- Visualized data distributions for key fields.
- Next steps may include deeper analyses of clinical variables, molecular status, and anatomical distribution based on the dataset's structure.